In [ ]:
import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## all trials

### init and viz

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id)

In [ ]:
encoder.verify()

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

In [ ]:
encoder.view_peths()

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()  # maximal explained variance
se.plot_dr2()  # unique explained variance

### encoder attributes

#### "raw" data

In [ ]:
print(encoder.spike_times.keys())

assert len(encoder.spike_times["DLS"]) == len(
    encoder.psths["DLS"]
)  # first axis is neuron count

In [ ]:
print(encoder.trial_data.shape)

assert encoder.trial_data.shape[0] == encoder.num_trials  # (num trials, -1)

In [ ]:
for k, psths_reg in encoder.psths.items():
    print(k, psths_reg.shape)  # (num units, num trials, num bins)

#### encoder io

In [ ]:
# tents (i.e., drift)
assert encoder.tents.shape == (encoder.num_trials, encoder.num_tents)

plt.figure(tight_layout=True)
plt.imshow(encoder.tents, aspect="auto")
plt.xlabel("tent basis fns")
plt.ylabel("trials")
plt.show()

In [ ]:
# tvs (i.e., binary task variables)
assert encoder.tvs.shape == (encoder.num_trials, encoder.num_tv)

plt.figure(tight_layout=True)
plt.imshow(encoder.tvs, aspect="auto")
plt.xlabel("task variables")
plt.ylabel("trials")
plt.show()

In [ ]:
# dm (i.e., design matrix, tents & tvs)
import numpy as np

assert encoder.dm.shape == (encoder.num_trials, encoder.num_tents + encoder.num_tv)
assert np.all(encoder.dm == np.hstack((encoder.tents, encoder.tvs)))

print(encoder.dm_names)

plt.figure(tight_layout=True)
plt.imshow(encoder.dm, aspect="auto")
plt.xlabel("regressors")
plt.ylabel("trials")
plt.show()

In [ ]:
# robs (i.e., r_obs, observed spike counts, normalized by default)

assert encoder.robs.shape == (encoder.num_trials, encoder.num_units)

plt.figure(tight_layout=True)
plt.imshow(encoder.robs, aspect="auto")
plt.xlabel("neurons")
plt.ylabel("trials")
plt.colorbar()
plt.show()

In [ ]:
# region idxs
assert np.all(
    np.sort(np.concatenate([idxs for idxs in encoder.reg_idxs.values()]))
    == np.arange(encoder.num_units)
)
encoder.reg_idxs

#### fit and predict

In [ ]:
encoder.fit_baseline()
encoder.baseline_predict()

encoder.fit_encoder()
encoder.encoder_predict()

In [ ]:
assert all(
    [
        encoder.robs_predict[k].shape == encoder.robs.shape
        for k in encoder.robs_predict.keys()
    ]
)

encoder.robs_predict.keys()

#### scores

In [ ]:
encoder.get_r2()

In [ ]:
from core.viz import plot_kdes

assert all(
    [encoder.scores[k].shape == (encoder.num_units,) for k in encoder.scores.keys()]
)

plot_kdes(encoder.scores, xlim=[-0.8, 1], label=r"$r^2$")

#### beta weights

In [ ]:
encoder.fit_encoder()

In [ ]:
assert encoder.encoder_weights.shape == (
    encoder.num_units,
    encoder.num_tents + encoder.num_tv,
)

plt.figure(tight_layout=True)
plt.imshow(encoder.encoder_weights, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
plt.xlabel("regressors")
plt.ylabel("neurons")
plt.colorbar()
plt.show()

In [ ]:
response_weights = encoder.get_weights(regr="response")

assert response_weights.shape == (
    encoder.num_units,
    2,
)  # 2 for the response_right and response_left
assert np.all(
    response_weights
    == encoder.encoder_weights[
        :, [encoder.dm_idxs[regr] for regr in ["response_left", "response_right"]]
    ]
)

## strategy split

### init and viz

In [ ]:
from sg.models import StrategyEncoder

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

In [ ]:
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder_mb.view_fits()

In [ ]:
encoder_mf.view_fits()

In [ ]:
encoder_mb.view_weights()

In [ ]:
encoder_mf.view_weights()

In [ ]:
encoder_mb.view_peths()

In [ ]:
encoder_mf.view_peths()

### encoder attributes

In [ ]:
encoders = [encoder_mb, encoder_mf]

#### "raw" data

In [ ]:
for encoder_ in encoders:
    assert encoder_.trial_data.shape[0] == encoder_.num_trials
    assert encoder_.num_trials == min(
        sum(encoder.trial_data["strategy"] == 1),
        sum(encoder.trial_data["strategy"] == -1),
    )
assert encoder_mb.trial_data.shape == encoder_mf.trial_data.shape

#### encoder io

In [ ]:
for encoder_ in encoders:
    assert encoder_.tents.shape == (encoder_.num_trials, encoder_.num_tents)
    assert encoder_.tvs.shape == (encoder_.num_trials, encoder_.num_tv)
    assert encoder_.dm.shape == (
        encoder_.num_trials,
        encoder_.num_tents + encoder_.num_tv,
    )
    assert encoder_.robs.shape == (encoder_.num_trials, encoder_.num_units)

#### fit and predict

In [ ]:
for encoder_ in encoders:
    encoder_.fit_baseline()
    encoder_.baseline_predict()

    encoder_.fit_encoder()
    encoder_.encoder_predict()

    assert all(
        [
            encoder_.robs_predict[k].shape == encoder_.robs.shape
            for k in encoder_.robs_predict.keys()
        ]
    )

#### scores

In [ ]:
for encoder_ in encoders:
    encoder_.get_r2()

    assert all(
        [
            encoder_.scores[k].shape == (encoder_.num_units,)
            for k in encoder_.scores.keys()
        ]
    )

#### beta weights

In [ ]:
for encoder_ in encoders:
    encoder_.fit_encoder()

    assert encoder_.encoder_weights.shape == (
        encoder_.num_units,
        encoder_.num_tents + encoder_.num_tv,
    )